# 00 — Generate Bank Data

One-time setup that writes raw multi-format files to `project/landing/`. Run this once before any other notebook in the project.

## What it produces (~5k rows total)

| Table | Rows | Format | Why this format |
|---|---|---|---|
| `customers` | 100 | CSV | Customer master often comes from a relational dump |
| `card_accounts` | 150 | CSV | Same — cards are issued by the core banking system |
| `card_transactions` | 2,000 | Parquet | High-volume event stream — columnar storage saves space |
| `loan_accounts` | 60 | JSON | Loans system exports nested JSON snapshots |
| `loan_repayments` | 500 | CSV | Repayment ledger from the EMI processor |
| `bank_accounts` | 120 | CSV | Core banking export |
| `account_transactions` | 1,500 | Parquet | High-volume |
| `payments` | 800 | JSON | UPI/NEFT switch sends JSON events |

## Deliberate raw-source quirks (silver layer cleans these up)

- **Loans use `customer_id` as INTEGER** (1, 2, 3, …) while everything else uses STRING (`C001`, `C002`).
- **Payments has no `customer_id`** at all — only `sender_account_id` and `receiver_account_id`. Silver derives `sender_customer_id` by joining to `bank_accounts`.
- A handful of rows have nulls and a few duplicates, so silver de-duplication is meaningful.

## Setup

In [ ]:
# Make project/conf importable whether we launch jupyter from project/ or the repo root
import sys
from pathlib import Path

if (Path.cwd() / "conf").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd() / "project" / "conf").exists():
    PROJECT_ROOT = Path.cwd() / "project"
else:
    raise RuntimeError("Run this notebook from inside project/ or the repo root")

sys.path.insert(0, str(PROJECT_ROOT))
print(f"PROJECT_ROOT: {PROJECT_ROOT}")

In [ ]:
import csv
import json
import random
import shutil
from datetime import date, datetime, timedelta
from decimal import Decimal

from conf.spark import get_spark, LANDING_DIR

RNG = random.Random(42)  # reproducible

# Reset landing dir so re-runs are clean
if LANDING_DIR.exists():
    shutil.rmtree(LANDING_DIR)
LANDING_DIR.mkdir(parents=True)
print(f"LANDING_DIR: {LANDING_DIR}")

In [ ]:
# Reference pools — kept small so generated data feels human
FIRST_NAMES = ["Priya", "Arjun", "Sneha", "Rohan", "Anjali", "Vikram", "Kavya",
               "Karan", "Meera", "Aditya", "Isha", "Rahul", "Pooja", "Sanjay",
               "Divya", "Nikhil", "Tara", "Aman", "Riya", "Suresh"]
LAST_NAMES  = ["Sharma", "Mehta", "Kapoor", "Reddy", "Iyer", "Patel", "Singh",
               "Gupta", "Bose", "Joshi", "Nair", "Khanna", "Rao", "Malhotra"]
CITIES = [("Mumbai", "Maharashtra"), ("Bangalore", "Karnataka"), ("Delhi", "Delhi"),
          ("Pune", "Maharashtra"), ("Hyderabad", "Telangana"), ("Chennai", "Tamil Nadu"),
          ("Kolkata", "West Bengal"), ("Ahmedabad", "Gujarat"), ("Jaipur", "Rajasthan")]
MERCHANTS = [
    ("BigBasket", "grocery"), ("DMart", "grocery"), ("More Megastore", "grocery"),
    ("MakeMyTrip", "travel"), ("IRCTC", "travel"), ("Indigo Air", "travel"),
    ("Shell", "fuel"), ("HP Petrol", "fuel"), ("Indian Oil", "fuel"),
    ("Amazon", "ecommerce"), ("Flipkart", "ecommerce"), ("Myntra", "ecommerce"),
    ("Swiggy", "food"), ("Zomato", "food"), ("Pizza Hut", "food"),
    ("Apollo Pharmacy", "health"), ("PharmEasy", "health"),
    ("Netflix", "entertainment"), ("BookMyShow", "entertainment"),
    ("Unknown Merch", "other"),  # shows up disproportionately in fraud rows
]

# A fixed "today" for reproducible date ranges
TODAY = date(2025, 1, 15)
TXN_WINDOW_DAYS = 90  # transactions span the last 90 days

## customers (CSV)

100 customers, IDs `C001`..`C100`. Written as CSV — the typical core-banking dump format.

In [ ]:
N_CUSTOMERS = 100

customers = []
for i in range(1, N_CUSTOMERS + 1):
    cid = f"C{i:03d}"
    fn = RNG.choice(FIRST_NAMES)
    ln = RNG.choice(LAST_NAMES)
    city, state = RNG.choice(CITIES)
    dob = date(RNG.randint(1970, 2000), RNG.randint(1, 12), RNG.randint(1, 28))
    created_days_ago = RNG.randint(180, 1500)
    created = datetime.combine(TODAY - timedelta(days=created_days_ago),
                                datetime.min.time()).replace(
        hour=RNG.randint(8, 20), minute=RNG.randint(0, 59))
    customers.append({
        "customer_id":   cid,
        "full_name":     f"{fn} {ln}",
        "email":         f"{fn.lower()}.{ln.lower()}{i}@example.com",
        "phone":         f"+91-9{RNG.randint(100000000, 999999999)}",
        "date_of_birth": dob.isoformat(),
        "gender":        RNG.choice(["M", "F", "F", "M", "Other"]),
        "city":          city,
        "state":         state,
        "country":       "IN",
        "created_at":    created.isoformat(sep=" "),
    })

# Inject a couple of duplicates and a row with a missing email — silver should handle
customers.append(dict(customers[0]))                   # exact duplicate of C001
customers.append({**customers[1], "email": ""})       # missing email

out = LANDING_DIR / "customers.csv"
with out.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(customers[0].keys()))
    w.writeheader()
    w.writerows(customers)
print(f"wrote {len(customers)} rows -> {out}")

## card_accounts (CSV) and card_transactions (Parquet)

Each customer gets 1–3 cards. ~3% of transactions are flagged (large amount, odd hour, often `Unknown Merch`).

In [ ]:
N_CARDS = 150
N_TXNS = 2000

card_accounts = []
for i in range(1, N_CARDS + 1):
    cid = f"C{RNG.randint(1, N_CUSTOMERS):03d}"
    card_type = RNG.choice(["credit", "credit", "debit"])
    card_accounts.append({
        "card_id":      f"CA{i:04d}",
        "customer_id":  cid,
        "card_type":    card_type,
        "credit_limit": str(RNG.choice([50000, 100000, 200000, 500000])) if card_type == "credit" else "",
        "status":       RNG.choice(["active"] * 9 + ["blocked", "closed"]),
        "issued_at":    (TODAY - timedelta(days=RNG.randint(30, 1200))).isoformat() + " 10:00:00",
    })

out = LANDING_DIR / "card_accounts.csv"
with out.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(card_accounts[0].keys()))
    w.writeheader()
    w.writerows(card_accounts)
print(f"wrote {len(card_accounts)} rows -> {out}")

In [ ]:
# card_transactions — write via Spark to Parquet
spark = get_spark("GenerateBankData")

txns = []
for i in range(1, N_TXNS + 1):
    card = RNG.choice(card_accounts)
    is_fraud = RNG.random() < 0.03
    if is_fraud:
        merchant, category = RNG.choice([("Unknown Merch", "other")] * 3 + MERCHANTS)
        amount = Decimal(str(RNG.randint(20000, 100000)))
        hour = RNG.choice([0, 1, 2, 3, 23])  # odd hours
    else:
        merchant, category = RNG.choice(MERCHANTS)
        amount = Decimal(str(RNG.randint(50, 8000))) + Decimal("0." + f"{RNG.randint(0,99):02d}")
        hour = RNG.randint(7, 22)
    days_ago = RNG.randint(0, TXN_WINDOW_DAYS)
    when = datetime.combine(TODAY - timedelta(days=days_ago), datetime.min.time()).replace(
        hour=hour, minute=RNG.randint(0, 59), second=RNG.randint(0, 59))
    txns.append((
        f"T{i:06d}",
        card["card_id"],
        card["customer_id"],
        merchant,
        category,
        amount,
        "INR",
        when,
        RNG.choice(["approved"] * 19 + ["declined"]),
        is_fraud,
    ))

from pyspark.sql.types import (StructType, StructField, StringType,
                                DecimalType, TimestampType, BooleanType)
schema = StructType([
    StructField("transaction_id",    StringType(), False),
    StructField("card_id",           StringType(), False),
    StructField("customer_id",       StringType(), False),
    StructField("merchant_name",     StringType()),
    StructField("merchant_category", StringType()),
    StructField("amount",            DecimalType(18, 2), False),
    StructField("currency",          StringType()),
    StructField("transaction_at",    TimestampType(), False),
    StructField("status",            StringType()),
    StructField("is_flagged",        BooleanType()),
])
df = spark.createDataFrame(txns, schema=schema)
out = LANDING_DIR / "card_transactions"
df.coalesce(1).write.mode("overwrite").parquet(str(out))
print(f"wrote {df.count()} rows -> {out}")

## loan_accounts (JSON) and loan_repayments (CSV)

**Quirk:** raw `customer_id` is **INTEGER** here (1..100) instead of STRING (`C001`). Silver fixes this.

In [ ]:
N_LOANS = 60

loan_accounts = []
for i in range(1, N_LOANS + 1):
    loan_type = RNG.choice(["personal", "home", "auto"])
    if loan_type == "home":
        principal = RNG.randint(2000000, 8000000)
        rate = round(RNG.uniform(7.0, 9.5), 2)
        tenure = RNG.choice([120, 180, 240])
    elif loan_type == "auto":
        principal = RNG.randint(400000, 1500000)
        rate = round(RNG.uniform(8.5, 11.0), 2)
        tenure = RNG.choice([36, 48, 60])
    else:
        principal = RNG.randint(50000, 800000)
        rate = round(RNG.uniform(11.0, 16.0), 2)
        tenure = RNG.choice([12, 24, 36])
    loan_accounts.append({
        "loan_id":          f"L{i:04d}",
        "customer_id":      RNG.randint(1, N_CUSTOMERS),  # INTEGER on purpose
        "loan_type":        loan_type,
        "principal_amount": principal,
        "interest_rate":    rate,
        "tenure_months":    tenure,
        "disbursed_at":     (TODAY - timedelta(days=RNG.randint(60, 700))).isoformat() + "T10:00:00",
        "status":           RNG.choice(["active"] * 8 + ["closed", "defaulted"]),
    })

out = LANDING_DIR / "loan_accounts.json"
with out.open("w") as f:
    for row in loan_accounts:
        f.write(json.dumps(row) + "\n")  # JSONL
print(f"wrote {len(loan_accounts)} rows -> {out}")

In [ ]:
# loan_repayments — ~8 EMIs per loan, mix of paid/partial/missed/waived
loan_repayments = []
rid = 0
for loan in loan_accounts:
    n_emis = RNG.randint(4, 12)
    disbursed = date.fromisoformat(loan["disbursed_at"][:10])
    emi_amount = Decimal(loan["principal_amount"]) / Decimal(loan["tenure_months"])
    emi_amount = emi_amount.quantize(Decimal("0.01"))
    for k in range(n_emis):
        rid += 1
        due = disbursed + timedelta(days=30 * (k + 1))
        roll = RNG.random()
        if roll < 0.70:
            status = "paid"
            paid_amount = emi_amount
            paid_date = due + timedelta(days=RNG.randint(0, 3))
        elif roll < 0.85:
            status = "partial"
            paid_amount = (emi_amount * Decimal("0.5")).quantize(Decimal("0.01"))
            paid_date = due + timedelta(days=RNG.randint(0, 5))
        elif roll < 0.97:
            status = "missed"
            paid_amount = None
            paid_date = None
        else:
            status = "waived"
            paid_amount = Decimal("0")
            paid_date = due
        loan_repayments.append({
            "repayment_id": f"R{rid:05d}",
            "loan_id":      loan["loan_id"],
            "customer_id":  loan["customer_id"],  # also INTEGER, matching loan source
            "due_date":     due.isoformat(),
            "paid_date":    paid_date.isoformat() if paid_date else "",
            "due_amount":   str(emi_amount),
            "paid_amount":  str(paid_amount) if paid_amount is not None else "",
            "status":       status,
        })

out = LANDING_DIR / "loan_repayments.csv"
with out.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(loan_repayments[0].keys()))
    w.writeheader()
    w.writerows(loan_repayments)
print(f"wrote {len(loan_repayments)} rows -> {out}")

## bank_accounts (CSV) and account_transactions (Parquet)

`balance_after` is computed as a true running balance per account, so the SQL window-function exercise in notebook 04 can recompute it and verify.

In [ ]:
N_ACCOUNTS = 120
N_ACCT_TXNS = 1500

bank_accounts = []
for i in range(1, N_ACCOUNTS + 1):
    acct_type = RNG.choice(["savings"] * 6 + ["current"] * 3 + ["fixed_deposit"])
    bank_accounts.append({
        "account_id":    f"A{i:04d}",
        "customer_id":   f"C{RNG.randint(1, N_CUSTOMERS):03d}",
        "account_type":  acct_type,
        "balance":       "0.00",  # filled in after txns are simulated
        "interest_rate": str(round(RNG.uniform(6.0, 7.5), 2)) if acct_type == "fixed_deposit" else "",
        "opened_at":     (TODAY - timedelta(days=RNG.randint(60, 1500))).isoformat() + " 09:00:00",
        "status":        RNG.choice(["active"] * 9 + ["closed"]),
    })

# Generate account_transactions, threading through running balance per account
balances = {a["account_id"]: Decimal(RNG.randint(5000, 50000)) for a in bank_accounts}
acct_txns = []
for i in range(1, N_ACCT_TXNS + 1):
    acct = RNG.choice(bank_accounts)
    aid = acct["account_id"]
    txn_type = RNG.choice(["deposit"] * 4 + ["withdrawal"] * 5 + ["interest", "fee"])
    amount = Decimal(str(RNG.randint(100, 25000))) + Decimal("0." + f"{RNG.randint(0,99):02d}")
    if txn_type in ("withdrawal", "fee"):
        balances[aid] = balances[aid] - amount
    else:
        balances[aid] = balances[aid] + amount
    when = datetime.combine(TODAY - timedelta(days=RNG.randint(0, TXN_WINDOW_DAYS)),
                            datetime.min.time()).replace(
        hour=RNG.randint(0, 23), minute=RNG.randint(0, 59))
    acct_txns.append((
        f"AT{i:06d}",
        aid,
        acct["customer_id"],
        txn_type,
        amount,
        balances[aid],
        when,
    ))

# Update bank_accounts.balance to the final running balance
for a in bank_accounts:
    a["balance"] = str(balances[a["account_id"]])

out = LANDING_DIR / "bank_accounts.csv"
with out.open("w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(bank_accounts[0].keys()))
    w.writeheader()
    w.writerows(bank_accounts)
print(f"wrote {len(bank_accounts)} rows -> {out}")

from pyspark.sql.types import StructType, StructField, StringType, DecimalType, TimestampType
acct_schema = StructType([
    StructField("txn_id",        StringType(), False),
    StructField("account_id",    StringType(), False),
    StructField("customer_id",   StringType(), False),
    StructField("txn_type",      StringType()),
    StructField("amount",        DecimalType(18, 2), False),
    StructField("balance_after", DecimalType(18, 2)),
    StructField("txn_at",        TimestampType(), False),
])
df_acct = spark.createDataFrame(acct_txns, schema=acct_schema)
out = LANDING_DIR / "account_transactions"
df_acct.coalesce(1).write.mode("overwrite").parquet(str(out))
print(f"wrote {df_acct.count()} rows -> {out}")

## payments (JSON)

**Quirk:** no `customer_id` in the raw payload — only `sender_account_id` and `receiver_account_id`. Silver derives `sender_customer_id` by joining to `bank_accounts`.

In [ ]:
N_PAYMENTS = 800

payments = []
account_ids = [a["account_id"] for a in bank_accounts]
for i in range(1, N_PAYMENTS + 1):
    sender = RNG.choice(account_ids)
    receiver = RNG.choice(account_ids)
    while receiver == sender:
        receiver = RNG.choice(account_ids)
    initiated = datetime.combine(TODAY - timedelta(days=RNG.randint(0, TXN_WINDOW_DAYS)),
                                  datetime.min.time()).replace(
        hour=RNG.randint(0, 23), minute=RNG.randint(0, 59))
    roll = RNG.random()
    if roll < 0.80:
        status = "success"
        settled = (initiated + timedelta(seconds=RNG.randint(2, 60))).isoformat(sep=" ")
        failure_reason = None
    elif roll < 0.95:
        status = "failed"
        settled = None
        failure_reason = RNG.choice(["insufficient_funds", "timeout", "invalid_account"])
    else:
        status = "reversed"
        settled = (initiated + timedelta(seconds=RNG.randint(2, 60))).isoformat(sep=" ")
        failure_reason = "reversal_requested"
    payments.append({
        "payment_id":          f"P{i:05d}",
        "sender_account_id":   sender,
        "receiver_account_id": receiver,
        # NOTE: no customer_id field — silver derives it
        "payment_mode":        RNG.choice(["UPI"] * 6 + ["NEFT", "RTGS", "IMPS"]),
        "amount":              str(RNG.randint(100, 100000)) + "." + f"{RNG.randint(0,99):02d}",
        "currency":            "INR",
        "initiated_at":        initiated.isoformat(sep=" "),
        "settled_at":          settled,
        "status":              status,
        "failure_reason":      failure_reason,
    })

out = LANDING_DIR / "payments.json"
with out.open("w") as f:
    for row in payments:
        f.write(json.dumps(row) + "\n")
print(f"wrote {len(payments)} rows -> {out}")

## Summary

In [ ]:
import os
for entry in sorted(LANDING_DIR.iterdir()):
    if entry.is_dir():
        size = sum(p.stat().st_size for p in entry.rglob("*") if p.is_file())
        print(f"{entry.name:30s}  (dir)  {size:>10,} bytes")
    else:
        print(f"{entry.name:30s}        {entry.stat().st_size:>10,} bytes")

print("\nAll raw bank data written. Continue with 01-bronze-ingest.ipynb")